In [29]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ishaans23/final-classifier-artifact/head_training_history.csv
/kaggle/input/datasets/ishaans23/final-classifier-artifact/layer4_training_history.csv
/kaggle/input/datasets/ishaans23/final-classifier-artifact/model_metadata.json
/kaggle/input/datasets/ishaans23/final-classifier-artifact/resnet50_layer4_best.pth
/kaggle/input/datasets/ishaans23/final-classifier-artifact/training_summary.json
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/cache_info.json
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/excluded_mass_cases.csv
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/image_metadata.csv
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/shared_mass_calc_audit.csv
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/masks/P_00001_LEFT_CC.png
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/masks/P_00001_LEFT_MLO.png
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/masks/P_00004_LEFT_CC.png
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/mas

In [30]:
from pathlib import Path
import json
import random

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

In [31]:
SEED = 42

TARGET_HEIGHT = 1024
TARGET_WIDTH = 640

BATCH_SIZE = 2
NUM_WORKERS = 0

BASE_CHANNELS = 32

MAX_EPOCHS = 25
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 1e-4

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-5

BCE_WEIGHT = 0.30
DICE_WEIGHT = 0.70

USE_AMP = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUTPUT_DIR = Path(
    "/kaggle/working/segmentation_training"
)

CHECKPOINT_DIR = (
    OUTPUT_DIR / "checkpoints"
)

FIGURE_DIR = (
    OUTPUT_DIR / "figures"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

BEST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "unet_best.pth"
)

LATEST_CHECKPOINT_PATH = (
    CHECKPOINT_DIR
    / "unet_latest.pth"
)

HISTORY_PATH = (
    OUTPUT_DIR
    / "training_history.csv"
)

print("Output:", OUTPUT_DIR)

Output: /kaggle/working/segmentation_training


In [32]:
KAGGLE_INPUT_ROOT = Path(
    "/kaggle/input"
)

dataset_candidates = []

for cache_info_path in KAGGLE_INPUT_ROOT.rglob(
    "cache_info.json"
):

    candidate_root = (
        cache_info_path.parent
    )

    required_paths = [
        candidate_root / "npy",
        candidate_root / "masks",
        candidate_root / "splits",
    ]

    if all(
        path.exists()
        for path in required_paths
    ):
        dataset_candidates.append(
            candidate_root
        )

assert len(dataset_candidates) > 0, (
    "Could not locate model-ready "
    "CBIS-DDSM dataset."
)

dataset_candidates = sorted(
    dataset_candidates,
    key=lambda path: len(str(path)),
)

PROJECT_ROOT = (
    dataset_candidates[0]
)

print("Dataset:")
print(PROJECT_ROOT)

Dataset:
/kaggle/input/datasets/ishaans23/cbis-ddsm-training


In [33]:
SPLITS_DIR = (
    PROJECT_ROOT / "splits"
)

train_df = pd.read_csv(
    SPLITS_DIR / "train.csv"
)

val_df = pd.read_csv(
    SPLITS_DIR / "val.csv"
)

print(
    "Training samples:",
    len(train_df)
)

print(
    "Validation samples:",
    len(val_df)
)

print(
    "\nTraining abnormality distribution:"
)

print(
    train_df[
        "abnormality_type"
    ].value_counts()
)

print(
    "\nValidation abnormality distribution:"
)

print(
    val_df[
        "abnormality_type"
    ].value_counts()
)

Training samples: 1885
Validation samples: 408

Training abnormality distribution:
abnormality_type
calcification         1027
mass                   821
mass+calcification      37
Name: count, dtype: int64

Validation abnormality distribution:
abnormality_type
calcification         215
mass                  185
mass+calcification      8
Name: count, dtype: int64


In [34]:
class RAMSegmentationDataset(Dataset):

    def __init__(
        self,
        images,
        masks,
        image_keys,
    ):
        self.images = images
        self.masks = masks
        self.image_keys = image_keys

    def __len__(self):

        return len(
            self.image_keys
        )

    def __getitem__(
        self,
        idx,
    ):

        return (
            self.images[idx],
            self.masks[idx],
            self.image_keys[idx],
        )

In [35]:
def load_segmentation_split_into_ram(
    dataframe,
    root_dir,
    split_name,
):

    num_samples = len(
        dataframe
    )

    images = np.empty(
        (
            num_samples,
            TARGET_HEIGHT,
            TARGET_WIDTH,
        ),
        dtype=np.uint16,
    )

    masks = np.empty(
        (
            num_samples,
            TARGET_HEIGHT,
            TARGET_WIDTH,
        ),
        dtype=np.uint8,
    )

    image_keys = (
        dataframe[
            "image_key"
        ].tolist()
    )


    for i, row in enumerate(
        tqdm(
            dataframe.itertuples(
                index=False
            ),
            total=num_samples,
            desc=(
                f"Loading {split_name} "
                "into RAM"
            ),
        )
    ):

        image_path = (
            root_dir
            / row.npy_path
        )

        mask_path = (
            root_dir
            / row.mask_path
        )


        image = np.load(
            image_path
        )

        mask = cv2.imread(
            str(mask_path),
            cv2.IMREAD_GRAYSCALE,
        )


        if image.shape != (
            TARGET_HEIGHT,
            TARGET_WIDTH,
        ):

            raise ValueError(
                f"{row.image_key}: "
                f"image shape "
                f"{image.shape}"
            )


        if mask is None:

            raise ValueError(
                f"{row.image_key}: "
                "mask could not be loaded"
            )


        if mask.shape != (
            TARGET_HEIGHT,
            TARGET_WIDTH,
        ):

            raise ValueError(
                f"{row.image_key}: "
                f"mask shape "
                f"{mask.shape}"
            )


        images[i] = image

        masks[i] = (
            mask > 0
        ).astype(
            np.uint8
        )


    images = torch.from_numpy(
        images
    )

    masks = torch.from_numpy(
        masks
    )


    image_gb = (
        images.numel()
        * images.element_size()
        / 1024**3
    )

    mask_gb = (
        masks.numel()
        * masks.element_size()
        / 1024**3
    )


    print(
        f"\n{split_name}: "
        f"{num_samples} samples"
    )

    print(
        "Images:",
        images.shape,
        images.dtype,
    )

    print(
        "Masks:",
        masks.shape,
        masks.dtype,
    )

    print(
        f"RAM usage: "
        f"{image_gb + mask_gb:.2f} GiB"
    )


    return RAMSegmentationDataset(
        images=images,
        masks=masks,
        image_keys=image_keys,
    )

In [36]:
train_dataset = (
    load_segmentation_split_into_ram(
        dataframe=train_df,
        root_dir=PROJECT_ROOT,
        split_name="Training",
    )
)

val_dataset = (
    load_segmentation_split_into_ram(
        dataframe=val_df,
        root_dir=PROJECT_ROOT,
        split_name="Validation",
    )
)

print(
    "\nDatasets loaded."
)

Loading Training into RAM:   0%|          | 0/1885 [00:00<?, ?it/s]


Training: 1885 samples
Images: torch.Size([1885, 1024, 640]) torch.uint16
Masks: torch.Size([1885, 1024, 640]) torch.uint8
RAM usage: 3.45 GiB


Loading Validation into RAM:   0%|          | 0/408 [00:00<?, ?it/s]


Validation: 408 samples
Images: torch.Size([408, 1024, 640]) torch.uint16
Masks: torch.Size([408, 1024, 640]) torch.uint8
RAM usage: 0.75 GiB

Datasets loaded.


In [37]:
PIN_MEMORY = (
    torch.cuda.is_available()
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print(
    "Training batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

Training batches: 943
Validation batches: 204


In [38]:
images, masks, image_keys = next(
    iter(train_loader)
)

print(
    "Images:",
    images.shape,
    images.dtype,
)

print(
    "Masks:",
    masks.shape,
    masks.dtype,
)

print(
    "Mask values:",
    torch.unique(masks)
)

print(
    "Keys:",
    image_keys
)

assert images.shape[1:] == (
    TARGET_HEIGHT,
    TARGET_WIDTH,
)

assert masks.shape == (
    images.shape
)

assert set(
    torch.unique(
        masks
    ).tolist()
).issubset(
    {0, 1}
)

print(
    "\nBatch verified."
)

Images: torch.Size([2, 1024, 640]) torch.uint16
Masks: torch.Size([2, 1024, 640]) torch.uint8
Mask values: tensor([0, 1], dtype=torch.uint8)
Keys: ('P_01292_RIGHT_CC', 'P_01349_LEFT_MLO')

Batch verified.


In [39]:
foreground_pixels = int(
    train_dataset.masks.sum().item()
)

total_pixels = (
    train_dataset.masks.numel()
)

background_pixels = (
    total_pixels
    - foreground_pixels
)

raw_pos_weight = (
    background_pixels
    / foreground_pixels
)

# Full raw weighting can be very aggressive
# for sparse segmentation.
#
# Use square-root balancing to retain emphasis
# on lesion pixels without overwhelming Dice loss.

POS_WEIGHT = float(
    np.sqrt(
        raw_pos_weight
    )
)

print(
    "Foreground pixels:",
    foreground_pixels,
)

print(
    "Background pixels:",
    background_pixels,
)

print(
    "Foreground fraction:",
    foreground_pixels
    / total_pixels,
)

print(
    "Raw positive weight:",
    raw_pos_weight,
)

print(
    "Used positive weight:",
    POS_WEIGHT,
)

Foreground pixels: 18744851
Background pixels: 1216608749
Foreground fraction: 0.015173672542015501
Raw positive weight: 64.9036233470194
Used positive weight: 8.056278504807254


In [40]:
device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device:",
    device
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


def prepare_segmentation_batch(
    images,
    masks,
    device,
):

    images = images.to(
        device,
        non_blocking=True,
    )

    masks = masks.to(
        device,
        non_blocking=True,
    )


    # uint16 -> float32 [0,1]
    images = (
        images.float()
        / 65535.0
    )

    # [B,H,W] -> [B,1,H,W]
    images = images.unsqueeze(1)


    masks = (
        masks.float()
        .unsqueeze(1)
    )


    return (
        images,
        masks,
    )

Device: cuda:0
GPU: Tesla T4


In [41]:
class DoubleConv(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
    ):
        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),

            nn.GroupNorm(
                num_groups=8,
                num_channels=out_channels,
            ),

            nn.ReLU(
                inplace=True
            ),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1,
                bias=False,
            ),

            nn.GroupNorm(
                num_groups=8,
                num_channels=out_channels,
            ),

            nn.ReLU(
                inplace=True
            ),
        )


    def forward(
        self,
        x,
    ):

        return self.block(x)

In [42]:
class UpBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
    ):
        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2,
        )

        self.conv = DoubleConv(
            out_channels
            + skip_channels,
            out_channels,
        )


    def forward(
        self,
        x,
        skip,
    ):

        x = self.up(x)


        if x.shape[-2:] != (
            skip.shape[-2:]
        ):

            x = F.interpolate(
                x,
                size=skip.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )


        x = torch.cat(
            [
                skip,
                x,
            ],
            dim=1,
        )


        return self.conv(x)

In [43]:
class UNet(nn.Module):

    def __init__(
        self,
        in_channels=1,
        out_channels=1,
        base_channels=32,
    ):
        super().__init__()


        self.enc1 = DoubleConv(
            in_channels,
            base_channels,
        )

        self.enc2 = DoubleConv(
            base_channels,
            base_channels * 2,
        )

        self.enc3 = DoubleConv(
            base_channels * 2,
            base_channels * 4,
        )

        self.enc4 = DoubleConv(
            base_channels * 4,
            base_channels * 8,
        )


        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2,
        )


        self.bottleneck = DoubleConv(
            base_channels * 8,
            base_channels * 16,
        )


        self.up4 = UpBlock(
            base_channels * 16,
            base_channels * 8,
            base_channels * 8,
        )

        self.up3 = UpBlock(
            base_channels * 8,
            base_channels * 4,
            base_channels * 4,
        )

        self.up2 = UpBlock(
            base_channels * 4,
            base_channels * 2,
            base_channels * 2,
        )

        self.up1 = UpBlock(
            base_channels * 2,
            base_channels,
            base_channels,
        )


        self.output = nn.Conv2d(
            base_channels,
            out_channels,
            kernel_size=1,
        )


    def forward(
        self,
        x,
    ):

        e1 = self.enc1(x)

        e2 = self.enc2(
            self.pool(e1)
        )

        e3 = self.enc3(
            self.pool(e2)
        )

        e4 = self.enc4(
            self.pool(e3)
        )


        bottleneck = (
            self.bottleneck(
                self.pool(e4)
            )
        )


        d4 = self.up4(
            bottleneck,
            e4,
        )

        d3 = self.up3(
            d4,
            e3,
        )

        d2 = self.up2(
            d3,
            e2,
        )

        d1 = self.up1(
            d2,
            e1,
        )


        return self.output(
            d1
        )

In [44]:
model = UNet(
    in_channels=1,
    out_channels=1,
    base_channels=BASE_CHANNELS,
)

model = model.to(
    device
)


trainable_parameters = sum(
    parameter.numel()
    for parameter
    in model.parameters()
    if parameter.requires_grad
)


print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)

Trainable parameters: 7,762,465


In [45]:
images, masks, _ = next(
    iter(train_loader)
)

images, masks = (
    prepare_segmentation_batch(
        images,
        masks,
        device,
    )
)


with torch.inference_mode():

    logits = model(
        images
    )


print(
    "Input:",
    images.shape
)

print(
    "Output:",
    logits.shape
)

print(
    "Target:",
    masks.shape
)


assert (
    logits.shape
    == masks.shape
)

print(
    "\nProduction U-Net "
    "shape verified."
)

Input: torch.Size([2, 1, 1024, 640])
Output: torch.Size([2, 1, 1024, 640])
Target: torch.Size([2, 1, 1024, 640])

Production U-Net shape verified.


In [46]:
pos_weight_tensor = torch.tensor(
    [POS_WEIGHT],
    dtype=torch.float32,
    device=device,
)

bce_loss = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor
)


def soft_dice_per_image(
    logits,
    targets,
    smooth=1.0,
):

    probabilities = (
        torch.sigmoid(
            logits
        )
    )

    dimensions = (
        1,
        2,
        3,
    )


    intersection = (
        probabilities
        * targets
    ).sum(
        dim=dimensions
    )


    denominator = (
        probabilities.sum(
            dim=dimensions
        )
        +
        targets.sum(
            dim=dimensions
        )
    )


    return (
        (
            2.0 * intersection
            + smooth
        )
        /
        (
            denominator
            + smooth
        )
    )


def segmentation_loss(
    logits,
    targets,
):

    bce = bce_loss(
        logits,
        targets,
    )

    soft_dice = (
        soft_dice_per_image(
            logits,
            targets,
        ).mean()
    )

    dice_loss = (
        1.0
        - soft_dice
    )

    total = (
        BCE_WEIGHT * bce
        +
        DICE_WEIGHT
        * dice_loss
    )


    return {
        "total": total,
        "bce": bce,
        "dice_loss": dice_loss,
        "soft_dice": soft_dice,
    }

In [47]:
def hard_segmentation_metrics(
    logits,
    targets,
    threshold=0.5,
    smooth=1.0,
):

    probabilities = (
        torch.sigmoid(
            logits
        )
    )

    predictions = (
        probabilities
        >= threshold
    ).float()


    dimensions = (
        1,
        2,
        3,
    )


    intersection = (
        predictions
        * targets
    ).sum(
        dim=dimensions
    )

    predicted_pixels = (
        predictions.sum(
            dim=dimensions
        )
    )

    target_pixels = (
        targets.sum(
            dim=dimensions
        )
    )


    dice = (
        2.0 * intersection
        + smooth
    ) / (
        predicted_pixels
        + target_pixels
        + smooth
    )


    union = (
        predicted_pixels
        + target_pixels
        - intersection
    )


    iou = (
        intersection
        + smooth
    ) / (
        union
        + smooth
    )


    return {
        "dice": dice,
        "iou": iou,
    }

In [48]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    scaler,
    device,
):

    model.train()

    running = {
        "loss": 0.0,
        "bce": 0.0,
        "dice_loss": 0.0,
        "soft_dice": 0.0,
        "hard_dice": 0.0,
        "iou": 0.0,
        "samples": 0,
    }


    for (
        images,
        masks,
        _,
    ) in tqdm(
        loader,
        desc="Training",
        leave=False,
    ):

        images, masks = (
            prepare_segmentation_batch(
                images,
                masks,
                device,
            )
        )

        batch_size = (
            images.shape[0]
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(
                USE_AMP
                and device.type
                == "cuda"
            ),
        ):

            logits = model(
                images
            )

            loss_parts = (
                segmentation_loss(
                    logits,
                    masks,
                )
            )


        scaler.scale(
            loss_parts["total"]
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()


        with torch.no_grad():

            hard_metrics = (
                hard_segmentation_metrics(
                    logits,
                    masks,
                )
            )


        running[
            "loss"
        ] += (
            loss_parts[
                "total"
            ].item()
            * batch_size
        )

        running[
            "bce"
        ] += (
            loss_parts[
                "bce"
            ].item()
            * batch_size
        )

        running[
            "dice_loss"
        ] += (
            loss_parts[
                "dice_loss"
            ].item()
            * batch_size
        )

        running[
            "soft_dice"
        ] += (
            loss_parts[
                "soft_dice"
            ].item()
            * batch_size
        )

        running[
            "hard_dice"
        ] += (
            hard_metrics[
                "dice"
            ].mean().item()
            * batch_size
        )

        running[
            "iou"
        ] += (
            hard_metrics[
                "iou"
            ].mean().item()
            * batch_size
        )

        running[
            "samples"
        ] += batch_size


    samples = (
        running["samples"]
    )


    return {
        key: (
            running[key]
            / samples
        )
        for key in [
            "loss",
            "bce",
            "dice_loss",
            "soft_dice",
            "hard_dice",
            "iou",
        ]
    }

In [49]:
def validate_one_epoch(
    model,
    loader,
    device,
):

    model.eval()

    running = {
        "loss": 0.0,
        "bce": 0.0,
        "dice_loss": 0.0,
        "soft_dice": 0.0,
        "hard_dice": 0.0,
        "iou": 0.0,
        "samples": 0,
    }


    with torch.inference_mode():

        for (
            images,
            masks,
            _,
        ) in tqdm(
            loader,
            desc="Validation",
            leave=False,
        ):

            images, masks = (
                prepare_segmentation_batch(
                    images,
                    masks,
                    device,
                )
            )

            batch_size = (
                images.shape[0]
            )


            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(
                    USE_AMP
                    and device.type
                    == "cuda"
                ),
            ):

                logits = model(
                    images
                )

                loss_parts = (
                    segmentation_loss(
                        logits,
                        masks,
                    )
                )


            hard_metrics = (
                hard_segmentation_metrics(
                    logits,
                    masks,
                )
            )


            running[
                "loss"
            ] += (
                loss_parts[
                    "total"
                ].item()
                * batch_size
            )

            running[
                "bce"
            ] += (
                loss_parts[
                    "bce"
                ].item()
                * batch_size
            )

            running[
                "dice_loss"
            ] += (
                loss_parts[
                    "dice_loss"
                ].item()
                * batch_size
            )

            running[
                "soft_dice"
            ] += (
                loss_parts[
                    "soft_dice"
                ].item()
                * batch_size
            )

            running[
                "hard_dice"
            ] += (
                hard_metrics[
                    "dice"
                ].mean().item()
                * batch_size
            )

            running[
                "iou"
            ] += (
                hard_metrics[
                    "iou"
                ].mean().item()
                * batch_size
            )

            running[
                "samples"
            ] += batch_size


    samples = (
        running["samples"]
    )


    return {
        key: (
            running[key]
            / samples
        )
        for key in [
            "loss",
            "bce",
            "dice_loss",
            "soft_dice",
            "hard_dice",
            "iou",
        ]
    }

In [50]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(
        USE_AMP
        and device.type
        == "cuda"
    ),
)

print(
    "Learning rate:",
    LEARNING_RATE
)

print(
    "Weight decay:",
    WEIGHT_DECAY
)

print(
    "AMP:",
    USE_AMP
)

Learning rate: 0.0003
Weight decay: 1e-05
AMP: True


In [51]:
def save_segmentation_checkpoint(
    path,
    epoch,
    model,
    optimizer,
    scaler,
    train_metrics,
    val_metrics,
):

    torch.save(
        {
            "epoch": epoch,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "scaler_state_dict":
                scaler.state_dict(),

            "train_metrics":
                train_metrics,

            "val_metrics":
                val_metrics,

            "configuration": {
                "resolution": [
                    TARGET_HEIGHT,
                    TARGET_WIDTH,
                ],
                "base_channels":
                    BASE_CHANNELS,
                "batch_size":
                    BATCH_SIZE,
                "learning_rate":
                    LEARNING_RATE,
                "weight_decay":
                    WEIGHT_DECAY,
                "bce_weight":
                    BCE_WEIGHT,
                "dice_weight":
                    DICE_WEIGHT,
                "positive_weight":
                    POS_WEIGHT,
                "seed":
                    SEED,
            },
        },
        path,
    )

In [52]:
history = []

best_val_soft_dice = (
    -float("inf")
)

epochs_without_improvement = 0


for epoch in range(
    1,
    MAX_EPOCHS + 1,
):

    print(
        f"\nEpoch "
        f"{epoch}/{MAX_EPOCHS}"
    )


    train_metrics = (
        train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scaler=scaler,
            device=device,
        )
    )


    val_metrics = (
        validate_one_epoch(
            model=model,
            loader=val_loader,
            device=device,
        )
    )


    row = {
        "epoch": epoch,

        "train_loss":
            train_metrics[
                "loss"
            ],

        "train_soft_dice":
            train_metrics[
                "soft_dice"
            ],

        "train_hard_dice":
            train_metrics[
                "hard_dice"
            ],

        "train_iou":
            train_metrics[
                "iou"
            ],

        "val_loss":
            val_metrics[
                "loss"
            ],

        "val_soft_dice":
            val_metrics[
                "soft_dice"
            ],

        "val_hard_dice":
            val_metrics[
                "hard_dice"
            ],

        "val_iou":
            val_metrics[
                "iou"
            ],
    }


    history.append(
        row
    )


    # Save history every epoch.
    pd.DataFrame(
        history
    ).to_csv(
        HISTORY_PATH,
        index=False,
    )


    print(
        f"Train Loss: "
        f"{train_metrics['loss']:.4f} | "
        f"Soft Dice: "
        f"{train_metrics['soft_dice']:.4f} | "
        f"Hard Dice: "
        f"{train_metrics['hard_dice']:.4f}"
    )

    print(
        f"Val Loss: "
        f"{val_metrics['loss']:.4f} | "
        f"Soft Dice: "
        f"{val_metrics['soft_dice']:.4f} | "
        f"Hard Dice: "
        f"{val_metrics['hard_dice']:.4f} | "
        f"IoU: "
        f"{val_metrics['iou']:.4f}"
    )


    save_segmentation_checkpoint(
        path=LATEST_CHECKPOINT_PATH,
        epoch=epoch,
        model=model,
        optimizer=optimizer,
        scaler=scaler,
        train_metrics=train_metrics,
        val_metrics=val_metrics,
    )


    improvement = (
        val_metrics[
            "soft_dice"
        ]
        >
        (
            best_val_soft_dice
            +
            EARLY_STOPPING_MIN_DELTA
        )
    )


    if improvement:

        best_val_soft_dice = (
            val_metrics[
                "soft_dice"
            ]
        )

        epochs_without_improvement = 0


        save_segmentation_checkpoint(
            path=BEST_CHECKPOINT_PATH,
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scaler=scaler,
            train_metrics=train_metrics,
            val_metrics=val_metrics,
        )


        print(
            "Saved new best model "
            f"(val soft Dice = "
            f"{best_val_soft_dice:.4f})"
        )


    else:

        epochs_without_improvement += 1


        print(
            "No improvement for "
            f"{epochs_without_improvement} "
            "epoch(s)."
        )


    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):

        print(
            "\nEarly stopping triggered."
        )

        break


Epoch 1/25


Training:   0%|          | 0/943 [00:00<?, ?it/s]

Validation:   0%|          | 0/204 [00:00<?, ?it/s]

Train Loss: 0.7621 | Soft Dice: 0.0507 | Hard Dice: 0.0469
Val Loss: 0.7462 | Soft Dice: 0.0635 | Hard Dice: 0.1157 | IoU: 0.0725
Saved new best model (val soft Dice = 0.0635)

Epoch 2/25


Training:   0%|          | 0/943 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/204 [00:00<?, ?it/s]

Train Loss: 0.6968 | Soft Dice: 0.1313 | Hard Dice: 0.1709
Val Loss: 0.6872 | Soft Dice: 0.1410 | Hard Dice: 0.1989 | IoU: 0.1260
No improvement for 4 epoch(s).

Epoch 16/25


Training:   0%|          | 0/943 [00:00<?, ?it/s]

Validation:   0%|          | 0/204 [00:00<?, ?it/s]

Train Loss: 0.6951 | Soft Dice: 0.1328 | Hard Dice: 0.1754
Val Loss: 0.6847 | Soft Dice: 0.1425 | Hard Dice: 0.1932 | IoU: 0.1217
No improvement for 5 epoch(s).

Early stopping triggered.


In [53]:
history_df = pd.read_csv(
    HISTORY_PATH
)

best_row = history_df.loc[
    history_df["val_soft_dice"].idxmax()
]

print("Best training epoch:")
print(best_row)

print(
    "\nBest validation soft Dice:",
    best_row["val_soft_dice"]
)

print(
    "Validation hard Dice at that epoch:",
    best_row["val_hard_dice"]
)

print(
    "Validation IoU at that epoch:",
    best_row["val_iou"]
)

Best training epoch:
epoch              11.000000
train_loss          0.702615
train_soft_dice     0.121219
train_hard_dice     0.165423
train_iou           0.103812
val_loss            0.688337
val_soft_dice       0.153190
val_hard_dice       0.194311
val_iou             0.124191
Name: 10, dtype: float64

Best validation soft Dice: 0.1531899576560439
Validation hard Dice at that epoch: 0.1943107617533974
Validation IoU at that epoch: 0.124191095969808


In [54]:
import json
import shutil
from pathlib import Path


TRAINING_ARTIFACT_DIR = Path(
    "/kaggle/working/segmentation_training_artifact"
)

TRAINING_ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# LOAD BEST CHECKPOINT METADATA
# ============================================================

best_checkpoint = torch.load(
    BEST_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)


training_summary = {
    "architecture": "U-Net",
    "base_channels": BASE_CHANNELS,
    "input_resolution": [
        TARGET_HEIGHT,
        TARGET_WIDTH,
    ],
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "bce_weight": BCE_WEIGHT,
    "dice_weight": DICE_WEIGHT,
    "positive_weight": POS_WEIGHT,
    "seed": SEED,

    "best_epoch": int(
        best_checkpoint["epoch"]
    ),

    "best_validation_soft_dice": float(
        best_checkpoint[
            "val_metrics"
        ]["soft_dice"]
    ),

    "validation_hard_dice_at_best_epoch": float(
        best_checkpoint[
            "val_metrics"
        ]["hard_dice"]
    ),

    "validation_iou_at_best_epoch": float(
        best_checkpoint[
            "val_metrics"
        ]["iou"]
    ),

    "checkpoint": "unet_best.pth",

    "notes": (
        "Checkpoint selected using validation soft Dice. "
        "Segmentation probability threshold has not yet "
        "been selected. Test set remains untouched."
    ),
}


with open(
    TRAINING_ARTIFACT_DIR
    / "training_summary.json",
    "w",
) as f:

    json.dump(
        training_summary,
        f,
        indent=2,
    )


# ============================================================
# COPY BEST CHECKPOINT
# ============================================================

shutil.copy2(
    BEST_CHECKPOINT_PATH,
    TRAINING_ARTIFACT_DIR
    / "unet_best.pth",
)


# ============================================================
# COPY TRAINING HISTORY
# ============================================================

shutil.copy2(
    HISTORY_PATH,
    TRAINING_ARTIFACT_DIR
    / "training_history.csv",
)


# ============================================================
# ZIP
# ============================================================

zip_path = shutil.make_archive(
    "/kaggle/working/"
    "segmentation_training_artifact",
    "zip",
    TRAINING_ARTIFACT_DIR,
)


print(
    json.dumps(
        training_summary,
        indent=2,
    )
)

print(
    "\nSaved:"
)

print(zip_path)

{
  "architecture": "U-Net",
  "base_channels": 32,
  "input_resolution": [
    1024,
    640
  ],
  "batch_size": 2,
  "learning_rate": 0.0003,
  "weight_decay": 1e-05,
  "bce_weight": 0.3,
  "dice_weight": 0.7,
  "positive_weight": 8.056278504807254,
  "seed": 42,
  "best_epoch": 11,
  "best_validation_soft_dice": 0.1531899576560439,
  "validation_hard_dice_at_best_epoch": 0.19431076175339745,
  "validation_iou_at_best_epoch": 0.124191095969808,
  "checkpoint": "unet_best.pth",
  "notes": "Checkpoint selected using validation soft Dice. Segmentation probability threshold has not yet been selected. Test set remains untouched."
}

Saved:
/kaggle/working/segmentation_training_artifact.zip
